# Thu thập và khám phá dữ liệu

**Phụ trách:** B. Bàn giao các bảng dùng chung theo [phân công](../PHAN_CONG.md).

## 1. Nguồn và phạm vi

Ghi nguồn tải, ngày tải, điều khoản sử dụng, phạm vi thời gian, cách ghi nhận và đơn vị quan sát của từng bảng.

Giải thích lý do nhóm chọn dữ liệu tai nạn giao thông và ba bảng hỗ trợ các câu hỏi nghiên cứu ra sao. Ghi công trình công khai liên quan và điều nhóm muốn khảo sát thêm.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Notebook có thể chạy từ thư mục gốc hoặc từ thư mục notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "environment.yml").is_file():
    raise FileNotFoundError("Mở Jupyter Notebook từ thư mục gốc của dự án.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

# Cho phép import src/ khi kernel chạy trong thư mục notebooks/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## 2. Khám phá dữ liệu (mục 3.2 của đề bài)

Khảo sát riêng ba bảng vụ tai nạn, phương tiện và người bị nạn. Đi sâu vào các cột phục vụ Q01–Q06, rồi tóm tắt ngắn các cột còn lại.

### 2.1. Tổng quan và kiểm tra cấu trúc

- Ghi số dòng, số cột của từng **tệp đầy đủ**, đơn vị quan sát và khóa dự kiến. Cell kiểm tra đầu vào bên dưới chỉ đọc mẫu nhỏ.
- Lập từ điển cho từng cột gồm ý nghĩa, kiểu đọc từ CSV, kiểu dùng khi phân tích, đơn vị hoặc bảng mã. Ghi cột cần chuyển kiểu hay bỏ và lý do.
- Đếm dòng trống, dòng trùng toàn bộ và dòng trùng khóa. Phân biệt bản ghi lặp với nhiều phương tiện hoặc nhiều người bị nạn thuộc cùng một vụ. Ghi quyết định giữ hoặc bỏ từng nhóm và số dòng bị ảnh hưởng.

[Đề bài CSC17104](../CSC17104_2026_Final_Project.pdf)

In [ ]:
expected_keys = {
    "Accidents0515.csv": {"Accident_Index"},
    "Vehicles0515.csv": {"Accident_Index", "Vehicle_Reference"},
    "Casualties0515.csv": {"Accident_Index", "Casualty_Reference", "Vehicle_Reference"},
}

for filename, keys in expected_keys.items():
    csv_path = raw_dir / filename
    if not csv_path.is_file():
        print(f"Chưa có {filename}. Tải CSV theo HUONG_DAN.md.")
        continue

    columns = pd.read_csv(csv_path, nrows=0, index_col=False).columns
    missing_keys = sorted(keys - set(columns))
    if missing_keys:
        raise ValueError(f"{filename} thiếu cột {missing_keys}")

    # Chỉ đọc các khóa có tên. Một số dòng nguồn có thêm trường cuối không tên.
    preview = pd.read_csv(
        csv_path, nrows=5, usecols=sorted(keys), index_col=False,
        dtype={"Accident_Index": "string"},
    )
    print(f"{filename}: {len(columns)} cột có tên, đọc được {len(preview)} dòng mẫu.")

In [ ]:
# Ghi lại shape và dtypes của từng bảng khi đã đọc toàn bộ tệp.
# Dùng duplicated() cho bản ghi trùng hoàn toàn. Kiểm tra khóa riêng
# theo đơn vị quan sát của từng bảng.

### 2.2. Cột định lượng

- Với các cột liên quan, chẳng hạn tuổi người bị nạn hoặc số phương tiện mỗi vụ, mô tả hình dạng phân phối, tâm (trung bình hoặc trung vị) và độ phân tán (độ lệch chuẩn hoặc IQR). Báo cả min/max và kiểm tra xem có hợp lý theo đơn vị và miền giá trị.
- Có thể dùng IQR hoặc z-score để đánh dấu điểm cần xem lại. Đối chiếu giá trị cực đoan với tài liệu và bối cảnh trước khi kết luận đó là bản ghi hợp lệ hay lỗi.
- Kiểm tra giá trị bất khả thi và mã giữ chỗ như tuổi âm, `-1` hoặc `999`. Xác minh ý nghĩa theo từng cột trước khi đổi mã hoặc loại dòng.

In [ ]:
# Chọn các cột đo lường, dùng describe() và vẽ phân phối cho cột liên quan.
# Ghi số dòng bị gắn cờ, lý do và quyết định sau khi kiểm tra ngoại lệ.

### 2.3. Cột phân loại

- Đếm số nhãn phân biệt, báo nhãn phổ biến và tần suất hoặc tỷ lệ. Kiểm tra mất cân bằng ở các nhóm quan trọng như loại đường, ánh sáng, phương tiện hoặc mức độ nghiêm trọng.
- Đối chiếu mã phân loại với từ điển. Kiểm tra khác biệt cách viết, khoảng trắng, lỗi chính tả hoặc nhiều mã tương đương. Cột lưu bằng số vẫn có thể là cột phân loại.
- Liệt kê nhãn hiếm, nêu tiêu chí và lý do nếu gộp nhóm. Báo cỡ mẫu sau khi chuẩn hóa để người đọc thấy tác động.

In [ ]:
# Dùng nunique() và value_counts(dropna=False) để rà số nhãn và tần suất.
# Xác minh ý nghĩa của từng mã trước khi chuẩn hóa hoặc gộp nhóm.

### 2.4. Dữ liệu thiếu

- Lập bảng số lượng và tỷ lệ thiếu **cho từng cột của từng bảng**, dựa trên số dòng tương ứng. Thống kê riêng ô trống và mã không rõ trước khi quy đổi.
- So sánh tỷ lệ thiếu theo năm, loại đường hoặc nhóm đối tượng liên quan. Ghi nguyên nhân có bằng chứng từ nguồn và đánh dấu các giả thuyết chưa xác minh.
- Nêu phương án theo cột, chẳng hạn bỏ dòng, điền giá trị hoặc tách thành nhóm riêng, cùng lý do và cỡ mẫu chịu ảnh hưởng. Với ML, học tham số điền thiếu trong pipeline trên tập huấn luyện.

In [ ]:
# Sau khi xác định mã thiếu, dùng isna().sum() và isna().mean()
# để lập bảng số lượng và tỷ lệ. So sánh độ thiếu giữa các nhóm.

### 2.5. Mối quan hệ giữa các biến

- Xem tương quan giữa những biến định lượng liên quan trên cùng đơn vị quan sát. Ghi số cặp có dữ liệu và giải thích cách chọn thước đo.
- Dùng bảng chéo và thống kê theo nhóm cho các tổ hợp có ý nghĩa, chẳng hạn loại đường với điều kiện ánh sáng, hoặc nhóm tuổi với loại phương tiện. Ghi số quan sát và mẫu số trước khi diễn giải tỷ lệ. Kiểm tra khóa và quan hệ ghép nếu dùng cột từ nhiều bảng.
- Nêu mối liên hệ bất ngờ, kiểm tra nhóm có ít dòng hoặc khác biệt do cơ cấu dữ liệu, rồi giải thích giới hạn diễn giải.

In [ ]:
# Dùng corr() cho biến định lượng, crosstab() hoặc groupby() cho
# tổ hợp phân loại và thống kê theo nhóm. Ghi mẫu số cho từng kết quả.

### 2.6. Tóm tắt khám phá

Viết **3–5 quan sát chính** sau khi chạy trên dữ liệu thật. Chỉ ra các vấn đề chất lượng nghiêm trọng, cách dự kiến tiền xử lý và những mẫu hình có thể dẫn đến hoặc giúp tinh chỉnh câu hỏi nghiên cứu.

## 3. Làm sạch

Thực hiện các quyết định từ phần khám phá. Ghi số dòng trước và sau mỗi thao tác thay đổi số dòng.

In [ ]:
# Viết phần kiểm tra chất lượng và chuẩn hóa từng bảng ở đây.

## 4. Tạo các bảng dùng chung cho Q1–Q6

Chạy cell dưới để chuẩn hóa khóa và mã STATS19, kiểm tra quan hệ ghép rồi xuất bốn bảng.
Ghi số dòng, dữ liệu thiếu, số khóa chưa ghép và mọi quyết định làm sạch trong phần khám phá.
`accident_counts.csv` có một dòng mỗi vụ. Hai bảng ghép giữ riêng cấp phương tiện và cấp người bị nạn.
Kiểm tra khóa người bị nạn bằng cả ba trường `accident_id`, `vehicle_ref` và `casualty_ref`; một số vụ dùng lại mã người bị nạn ở các xe khác nhau.
Chỉ đọc các cột có tên được xác minh trong tiêu đề. Cột IMD có thể trống tùy phiên bản dữ liệu. Ghi độ phủ trước khi thực hiện Q5.

[Câu hỏi nghiên cứu](../question.md) · [Bảng giao nhận và tên cột](../PHAN_CONG.md)

In [ ]:
from tempfile import TemporaryDirectory

from src.data_prep.road_safety import build_tables

required_raw = [
    raw_dir / "Accidents0515.csv",
    raw_dir / "Vehicles0515.csv",
    raw_dir / "Casualties0515.csv",
]
missing_raw = [path.name for path in required_raw if not path.is_file()]
if missing_raw:
    print("Chưa tạo bảng xử lý. Thiếu tệp:", ", ".join(missing_raw))
else:
    # Kiểm tra cả phép ghép trước khi ghi bất kỳ bảng nào.
    tables = build_tables(raw_dir)
    processed_dir.mkdir(parents=True, exist_ok=True)
    # Ghi đủ bốn CSV ở thư mục tạm trước khi thay các đầu ra cũ.
    with TemporaryDirectory(dir=processed_dir) as temp_dir:
        staging = Path(temp_dir)
        for filename, frame in tables.items():
            frame.to_csv(staging / filename, index=False, date_format="%Y-%m-%d")
        for filename, frame in tables.items():
            (staging / filename).replace(processed_dir / filename)
            print(f"{filename}: {len(frame):,} dòng, {len(frame.columns)} cột")
    print(tables["casualty_vehicle.csv"]["vehicle_match_status"].value_counts())
    print(tables["vehicle_accident.csv"]["accident_match"].value_counts())

## 5. Tra nhãn STATS19

Nếu gói nguồn có các tệp tra mã, đặt chúng dưới `data/contextCSVs/`. Kiểm tra tiêu đề cột trước khi ánh xạ. Nếu thiếu tệp hoặc có cấu trúc khác, đối chiếu bảng mã DfT và ghi nguồn phiên bản đã dùng.

In [ ]:
from src.data_prep.road_safety import read_lookup

context_dir = project_root / "data" / "contextCSVs"
for filename in ("Road_Type.csv", "Casualty_Type.csv", "Vehicle_Type.csv",
                 "1st_Point_of_Impact.csv"):
    lookup_path = context_dir / filename
    if not lookup_path.is_file():
        print(f"Chưa có bảng mã {filename}.")
        continue

    # Một số bản ghi Code/Label với khác biệt chữ hoa và khoảng trắng.
    headers = {name.strip().lower(): name for name in pd.read_csv(lookup_path, nrows=0)}
    if {"code", "label"} <= headers.keys():
        lookup = read_lookup(lookup_path, code_column=headers["code"],
                             label_column=headers["label"])
        print(f"{filename}: {len(lookup)} mã hợp lệ.")
    else:
        print(f"{filename}: chọn cột mã và nhãn từ {list(headers.values())}.")